# Przegląd danych


In [3]:
import pandas as pd
import numpy as np

# Wczytanie danych
df = pd.read_csv('Data.csv')

# Szybki przegląd
print("Kształt danych:", df.shape)
print("\nTypy kolumn:")
print(df.dtypes)
print("\nPierwsze wiersze:")
df.head()

Kształt danych: (777, 11)

Typy kolumn:
PO_ID                object
Supplier             object
Order_Date           object
Delivery_Date        object
Item_Category        object
Order_Status         object
Quantity              int64
Unit_Price          float64
Negotiated_Price    float64
Defective_Units     float64
Compliance           object
dtype: object

Pierwsze wiersze:


,PO_ID,Supplier,Order_Date,Delivery_Date,Item_Category,Order_Status,Quantity,Unit_Price,Negotiated_Price,Defective_Units,Compliance
0,PO-00001,Alpha_Inc,2023-10-17,2023-10-25,Office Supplies,Cancelled,1176,20.13,17.81,NaN,Yes
1,PO-00002,Delta_Logistics,2022-04-25,2022-05-05,Office Supplies,Delivered,1509,39.32,37.34,235.0,Yes
2,PO-00003,Gamma_Co,2022-01-26,2022-02-15,MRO,Delivered,910,95.51,92.26,41.0,Yes
3,PO-00004,Beta_Supplies,2022-10-09,2022-10-28,Packaging,Delivered,1344,99.85,95.52,112.0,Yes
4,PO-00005,Delta_Logistics,2022-09-08,2022-09-20,Raw Materials,Delivered,1180,64.07,60.53,171.0,No


In [4]:
# Braki danych
print("Braki danych (liczba i %):")
missing = pd.DataFrame({
    'braki': df.isnull().sum(),
    'procent': (df.isnull().sum() / len(df) * 100).round(2)
})
print(missing[missing['braki'] > 0])

# Rozkład statusów zamówień
print("\nOrder_Status:")
print(df['Order_Status'].value_counts())

# Czy braki w Delivery_Date pokrywają się z Pending/Cancelled?
print("\nBraki Delivery_Date wg statusu:")
print(df[df['Delivery_Date'].isnull()]['Order_Status'].value_counts())

# Unikalne kategorie i dostawcy
print("\nLiczba unikalnych dostawców:", df['Supplier'].nunique())
print("Liczba unikalnych kategorii:", df['Item_Category'].nunique())
print("Kategorie:", df['Item_Category'].unique())

Braki danych (liczba i %):
                 braki  procent
Delivery_Date       87     11.2
Defective_Units    136     17.5

Order_Status:
Order_Status
Delivered              560
Pending                 81
Partially Delivered     73
Cancelled               63
Name: count, dtype: int64

Braki Delivery_Date wg statusu:
Order_Status
Delivered              68
Cancelled               8
Pending                 6
Partially Delivered     5
Name: count, dtype: int64

Liczba unikalnych dostawców: 5
Liczba unikalnych kategorii: 5
Kategorie: ['Office Supplies' 'MRO' 'Packaging' 'Raw Materials' 'Electronics']


In [5]:
# Konwersja dat
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Delivery_Date'] = pd.to_datetime(df['Delivery_Date'])

# Sprawdźmy niespójności: Delivered bez daty dostawy
delivered_no_date = df[(df['Order_Status'] == 'Delivered') & (df['Delivery_Date'].isnull())]
print(f"Delivered bez daty dostawy: {len(delivered_no_date)}")

# Non-delivered z datą dostawy
non_delivered_with_date = df[(df['Order_Status'] != 'Delivered') & (df['Delivery_Date'].notnull())]
print(f"Nie-Delivered z datą dostawy: {len(non_delivered_with_date)}")
print(non_delivered_with_date['Order_Status'].value_counts())

# Czy są przypadki Delivery_Date wcześniejsze niż Order_Date? (błąd logiczny)
invalid_dates = df[df['Delivery_Date'] < df['Order_Date']]
print(f"\nDostawa przed zamówieniem (błąd): {len(invalid_dates)}")

# Sprawdźmy braki Defective_Units w relacji do statusu
print("\nBraki Defective_Units wg statusu:")
print(df[df['Defective_Units'].isnull()]['Order_Status'].value_counts())

# Zakres dat
print(f"\nZakres Order_Date: {df['Order_Date'].min()} do {df['Order_Date'].max()}")

Delivered bez daty dostawy: 68
Nie-Delivered z datą dostawy: 198
Order_Status
Pending                75
Partially Delivered    68
Cancelled              55
Name: count, dtype: int64

Dostawa przed zamówieniem (błąd): 1

Braki Defective_Units wg statusu:
Order_Status
Delivered              102
Partially Delivered     13
Pending                 12
Cancelled                9
Name: count, dtype: int64

Zakres Order_Date: 2022-01-01 00:00:00 do 2024-01-01 00:00:00


In [6]:
# Flaga błędu logicznego: dostawa przed zamówieniem
df['flag_invalid_date_order'] = df['Delivery_Date'] < df['Order_Date']

# Flaga niespójności statusu vs obecności daty
df['flag_delivered_no_date'] = (df['Order_Status'] == 'Delivered') & (df['Delivery_Date'].isnull())
df['flag_nondelivered_has_date'] = (df['Order_Status'] != 'Delivered') & (df['Delivery_Date'].notnull())

# Lead time liczony tam, gdzie mamy obie daty i nie ma błędu logicznego
df['Lead_Time_Days'] = np.where(
    df['flag_invalid_date_order'],
    np.nan,
    (df['Delivery_Date'] - df['Order_Date']).dt.days
)

# Usuwamy tylko ten 1 rekord z jawnym błędem logicznym (do osobnego pliku, nie kasujemy na zawsze)
df_errors = df[df['flag_invalid_date_order']].copy()
df = df[~df['flag_invalid_date_order']].copy()

print(f"Usunięto {len(df_errors)} rekord(y) z błędem logicznym dat.")
print(f"Pozostało: {len(df)} rekordów")

# Kluczowe metryki finansowe
df['Order_Value'] = df['Negotiated_Price'] * df['Quantity']
df['Savings_Amount'] = (df['Unit_Price'] - df['Negotiated_Price']) * df['Quantity']
df['Savings_Rate_%'] = ((df['Unit_Price'] - df['Negotiated_Price']) / df['Unit_Price'] * 100).round(2)

# Defect rate — tylko tam gdzie mamy dane
df['Defect_Rate_%'] = np.where(
    df['Defective_Units'].notnull(),
    (df['Defective_Units'] / df['Quantity'] * 100).round(2),
    np.nan
)

df.head()

Usunięto 1 rekord(y) z błędem logicznym dat.
Pozostało: 776 rekordów


,PO_ID,Supplier,Order_Date,Delivery_Date,Item_Category,Order_Status,Quantity,Unit_Price,Negotiated_Price,Defective_Units,Compliance,flag_invalid_date_order,flag_delivered_no_date,flag_nondelivered_has_date,Lead_Time_Days,Order_Value,Savings_Amount,Savings_Rate_%,Defect_Rate_%
0,PO-00001,Alpha_Inc,2023-10-17,2023-10-25,Office Supplies,Cancelled,1176,20.13,17.81,NaN,Yes,False,False,True,8.0,20944.56,2728.32,11.53,NaN
1,PO-00002,Delta_Logistics,2022-04-25,2022-05-05,Office Supplies,Delivered,1509,39.32,37.34,235.0,Yes,False,False,False,10.0,56346.06,2987.82,5.04,15.57
2,PO-00003,Gamma_Co,2022-01-26,2022-02-15,MRO,Delivered,910,95.51,92.26,41.0,Yes,False,False,False,20.0,83956.60,2957.50,3.40,4.51
3,PO-00004,Beta_Supplies,2022-10-09,2022-10-28,Packaging,Delivered,1344,99.85,95.52,112.0,Yes,False,False,False,19.0,128378.88,5819.52,4.34,8.33
4,PO-00005,Delta_Logistics,2022-09-08,2022-09-20,Raw Materials,Delivered,1180,64.07,60.53,171.0,No,False,False,False,12.0,71425.40,4177.20,5.53,14.49


In [7]:
# Walidacja Savings_Rate_%
print("Savings_Rate_% — statystyki:")
print(df['Savings_Rate_%'].describe())
print(f"\nUjemne savings (negocjacja podniosła cenę): {(df['Savings_Rate_%'] < 0).sum()}")
print(f"Savings_Rate_% == 0 (brak negocjacji): {(df['Savings_Rate_%'] == 0).sum()}")

# Walidacja Defect_Rate_%
print("\nDefect_Rate_% — statystyki:")
print(df['Defect_Rate_%'].describe())
print(f"\nDefect rate > 100% (błąd — więcej defektów niż zamówiono): {(df['Defect_Rate_%'] > 100).sum()}")

# Walidacja Lead_Time_Days
print("\nLead_Time_Days — statystyki:")
print(df['Lead_Time_Days'].describe())
print(f"\nUjemny lead time: {(df['Lead_Time_Days'] < 0).sum()}")

# Rozkład compliance
print("\nCompliance:")
print(df['Compliance'].value_counts())

Savings_Rate_% — statystyki:
count    776.000000
mean       7.960902
std        4.136469
min        1.010000
25%        4.277500
50%        7.900000
75%       11.530000
max       14.990000
Name: Savings_Rate_%, dtype: float64

Ujemne savings (negocjacja podniosła cenę): 0
Savings_Rate_% == 0 (brak negocjacji): 0

Defect_Rate_% — statystyki:
count    640.000000
mean       7.015922
std        5.005095
min        0.000000
25%        2.795000
50%        5.165000
75%       10.525000
max       35.710000
Name: Defect_Rate_%, dtype: float64

Defect rate > 100% (błąd — więcej defektów niż zamówiono): 0

Lead_Time_Days — statystyki:
count    689.000000
mean      10.799710
std        5.701688
min        1.000000
25%        6.000000
50%       11.000000
75%       16.000000
max       20.000000
Name: Lead_Time_Days, dtype: float64

Ujemny lead time: 0

Compliance:
Compliance
Yes    639
No     137
Name: count, dtype: int64


In [8]:
# Benchmark lead time per kategoria (mediana)
category_benchmark = df.groupby('Item_Category')['Lead_Time_Days'].median()
print("Mediana Lead_Time_Days per kategoria:")
print(category_benchmark)

# Mapujemy benchmark na każdy wiersz
df['Category_Lead_Time_Benchmark'] = df['Item_Category'].map(category_benchmark)

# Flaga on-time — tylko tam gdzie mamy Lead_Time_Days
df['Is_On_Time'] = np.where(
    df['Lead_Time_Days'].notnull(),
    df['Lead_Time_Days'] <= df['Category_Lead_Time_Benchmark'],
    np.nan
)

print(f"\nOn-time rate (ogółem): {df['Is_On_Time'].mean() * 100:.1f}%")

# Szybki podgląd KPI per dostawca — sprawdzian przed zapisem
supplier_summary = df.groupby('Supplier').agg(
    total_orders=('PO_ID', 'count'),
    total_spend=('Order_Value', 'sum'),
    avg_savings_rate=('Savings_Rate_%', 'mean'),
    avg_defect_rate=('Defect_Rate_%', 'mean'),
    on_time_rate=('Is_On_Time', 'mean'),
    compliance_rate=('Compliance', lambda x: (x == 'Yes').mean())
).round(2)
print("\nPodsumowanie per dostawca:")
print(supplier_summary)

Mediana Lead_Time_Days per kategoria:
Item_Category
Electronics        11.0
MRO                12.0
Office Supplies    10.0
Packaging          11.0
Raw Materials      10.0
Name: Lead_Time_Days, dtype: float64

On-time rate (ogółem): 52.2%

Podsumowanie per dostawca:
                 total_orders  total_spend  avg_savings_rate  avg_defect_rate  \
Supplier                                                                        
Alpha_Inc                 140   7825912.85              8.17             2.37   
Beta_Supplies             156   9858665.90              7.83             9.78   
Delta_Logistics           171   9236240.47              7.81            14.63   
Epsilon_Group             166   9851156.06              8.04             3.07   
Gamma_Co                  143   8587921.71              7.98             5.02   

                 on_time_rate  compliance_rate  
Supplier                                        
Alpha_Inc                0.53             0.94  
Beta_Supplies     

In [9]:
# Zapis oczyszczonych danych
df.to_csv('procurement_clean.csv', index=False)
print(f"Zapisano procurement_clean.csv — {df.shape[0]} wierszy, {df.shape[1]} kolumn")

# Pobranie pliku z Colaba na dysk (żeby wrzucić do repo na GitHub)
from google.colab import files
files.download('procurement_clean.csv')

Zapisano procurement_clean.csv — 776 wierszy, 21 kolumn


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>